In [0]:
# ==========================================
# 05 - DATA QUALITY TESTING
# ==========================================

from pyspark.sql import functions as F

transactions = spark.table(
    "fraud_detection.silver.transactions"
)

customers = spark.table(
    "fraud_detection.silver.customers"
)

merchants = spark.table(
    "fraud_detection.silver.merchants"
)

print("Silver datasets loaded successfully")

print("Transactions:", transactions.count())
print("Customers:", customers.count())
print("Merchants:", merchants.count())

In [0]:
# ==========================================
# CREATE INVALID TEST RECORDS
# ==========================================

from decimal import Decimal

valid_transaction = transactions.limit(1)

base_row = valid_transaction.first()

invalid_rows = [
    # Missing customer ID
    base_row.asDict() | {
        "transaction_id": "TEST-000001",
        "customer_id": None
    },

    # Invalid merchant ID
    base_row.asDict() | {
        "transaction_id": "TEST-000002",
        "merchant_id": "INVALID-MERCHANT"
    },

    # Negative transaction amount
    base_row.asDict() | {
        "transaction_id": "TEST-000003",
        "amount": Decimal("-500.00")
    },

    # Invalid fraud label
    base_row.asDict() | {
        "transaction_id": "TEST-000004",
        "fraud_label": 5
    },

    # Null transaction timestamp
    base_row.asDict() | {
        "transaction_id": "TEST-000005",
        "transaction_timestamp": None
    }
]

invalid_test_df = spark.createDataFrame(
    invalid_rows,
    schema=transactions.schema
)

print("Invalid test records:", invalid_test_df.count())

display(invalid_test_df)

In [0]:
# ==========================================
# RUN VALIDATION CHECKS
# ==========================================

# Referential-integrity key sets
valid_customers = customers.select(
    F.col("customer_id").alias("_valid_customer_id")
).distinct()

valid_merchants = merchants.select(
    F.col("merchant_id").alias("_valid_merchant_id")
).distinct()

checked_df = (
    invalid_test_df
    .join(valid_customers, on=F.col("customer_id") == F.col("_valid_customer_id"), how="left")
    .join(valid_merchants, on=F.col("merchant_id") == F.col("_valid_merchant_id"), how="left")
    .withColumn(
        "check_customer_id",
        F.col("customer_id").isNotNull() & F.col("_valid_customer_id").isNotNull(),
    )
    .withColumn(
        "check_merchant_id",
        F.col("merchant_id").isNotNull() & F.col("_valid_merchant_id").isNotNull(),
    )
    .withColumn(
        "check_amount",
        F.col("amount").isNotNull() & (F.col("amount") > 0),
    )
    .withColumn(
        "check_fraud_label",
        F.col("fraud_label").isNotNull()
        & F.coalesce(F.col("fraud_label").isin(0, 1), F.lit(False)),
    )
    .withColumn(
        "check_timestamp",
        F.col("transaction_timestamp").isNotNull(),
    )
)

is_valid = (
    F.col("check_customer_id")
    & F.col("check_merchant_id")
    & F.col("check_amount")
    & F.col("check_fraud_label")
    & F.col("check_timestamp")
)

quarantined_df = checked_df.filter(~is_valid)
valid_records_df = checked_df.filter(is_valid)

print("Records quarantined:", quarantined_df.count())
print("Records passed all checks:", valid_records_df.count())

display(
    quarantined_df.select(
        "transaction_id",
        "check_customer_id",
        "check_merchant_id",
        "check_amount",
        "check_fraud_label",
        "check_timestamp",
    )
)

# Expected outcome: every invalid test record is quarantined
assert quarantined_df.count() == 5, "Expected all 5 invalid records to be quarantined"
assert valid_records_df.count() == 0, "Expected no invalid records to pass the checks"
print("All validation checks passed")

In [0]:
# ==========================================
# TEST DUPLICATE DETECTION
# ==========================================

duplicate_row = base_row.asDict() | {
    "transaction_id": "TEST-000006"
}

duplicate_test_df = spark.createDataFrame(
    [duplicate_row, duplicate_row],
    schema=transactions.schema,
)

combined_test_df = invalid_test_df.unionByName(duplicate_test_df)

duplicates_df = (
    combined_test_df
    .groupBy("transaction_id")
    .agg(F.count("*").alias("occurrences"))
    .filter(F.col("occurrences") > 1)
)

print("Combined test records:", combined_test_df.count())
print("Duplicate transaction IDs found:", duplicates_df.count())

display(duplicates_df)

# Expected outcome: exactly one duplicated transaction ID is identified
assert duplicates_df.count() == 1, "Expected exactly one duplicate transaction ID"
assert duplicates_df.first()["occurrences"] == 2, "Expected the duplicate to occur twice"
print("Duplicate detection check passed")

In [0]:
# ==========================================
# VALIDATE REAL SILVER DATA
# ==========================================

# Apply the same checks to the full silver transactions table
silver_checked_df = (
    transactions
    .join(valid_customers, on=F.col("customer_id") == F.col("_valid_customer_id"), how="left")
    .join(valid_merchants, on=F.col("merchant_id") == F.col("_valid_merchant_id"), how="left")
    .withColumn(
        "check_customer_id",
        F.col("customer_id").isNotNull() & F.col("_valid_customer_id").isNotNull(),
    )
    .withColumn(
        "check_merchant_id",
        F.col("merchant_id").isNotNull() & F.col("_valid_merchant_id").isNotNull(),
    )
    .withColumn(
        "check_amount",
        F.col("amount").isNotNull() & (F.col("amount") > 0),
    )
    .withColumn(
        "check_fraud_label",
        F.col("fraud_label").isNotNull()
        & F.coalesce(F.col("fraud_label").isin(0, 1), F.lit(False)),
    )
    .withColumn(
        "check_timestamp",
        F.col("transaction_timestamp").isNotNull(),
    )
)

silver_all_valid = (
    F.col("check_customer_id")
    & F.col("check_merchant_id")
    & F.col("check_amount")
    & F.col("check_fraud_label")
    & F.col("check_timestamp")
)

silver_summary_df = silver_checked_df.agg(
    F.count("*").alias("total_records"),
    F.sum(F.when(~F.col("check_customer_id"), 1).otherwise(0)).alias("customer_id_failures"),
    F.sum(F.when(~F.col("check_merchant_id"), 1).otherwise(0)).alias("merchant_id_failures"),
    F.sum(F.when(~F.col("check_amount"), 1).otherwise(0)).alias("amount_failures"),
    F.sum(F.when(~F.col("check_fraud_label"), 1).otherwise(0)).alias("fraud_label_failures"),
    F.sum(F.when(~F.col("check_timestamp"), 1).otherwise(0)).alias("timestamp_failures"),
    F.sum(F.when(silver_all_valid, 1).otherwise(0)).alias("valid_records"),
)

print("Validation summary for fraud_detection.silver.transactions:")
display(silver_summary_df)

# Fail the pipeline if any real silver record is invalid
silver_summary = silver_summary_df.first()

assert silver_summary["total_records"] > 0, "Silver transactions table is empty"
assert silver_summary["valid_records"] == silver_summary["total_records"], (
    "Silver has data quality failures: "
    f"customer_id={silver_summary['customer_id_failures']}, "
    f"merchant_id={silver_summary['merchant_id_failures']}, "
    f"amount={silver_summary['amount_failures']}, "
    f"fraud_label={silver_summary['fraud_label_failures']}, "
    f"timestamp={silver_summary['timestamp_failures']}"
)

print("Silver data quality checks passed")

In [0]:
# ==========================================
# CHECK SILVER FOR DUPLICATES
# ==========================================

silver_duplicates_df = (
    transactions
    .groupBy("transaction_id")
    .agg(F.count("*").alias("occurrences"))
    .filter(F.col("occurrences") > 1)
)

duplicate_count = silver_duplicates_df.count()
print("Duplicate transaction IDs in silver:", duplicate_count)

if duplicate_count > 0:
    display(silver_duplicates_df.orderBy(F.col("occurrences").desc()))
else:
    print("No duplicate transaction IDs found - silver deduplication is working")

# Fail the pipeline if duplicates are found in silver
assert duplicate_count == 0, "Silver contains duplicate transaction IDs"
print("Silver duplicate detection check passed")